# Transformation to C Code (Simple)

## Imports

In [1]:
# | code-fold: true
# | code-summary: "Load packages"
# | output: false

import os
import numpy as np

from zoomy_core.model.models.shallow_water import ShallowWaterEquations as SWE
from zoomy_core.model.models.shallow_water_topo import NumericalShallowWaterEquationsWithTopo 
from zoomy_core.model.models.shallow_moments_topo import NumericalShallowMomentsTopo, ShallowMomentsTopo
from zoomy_core.fvm.symbolic_numerics import (
    PositiveRusanov,
    Rusanov,
    QuasilinearRusanov,
    PositiveQuasilinearRusanov,
    NonconservativeRusanov,
    PositiveNonconservativeRusanov,
)

import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
from zoomy_core.misc.misc import Zstruct, Settings
from zoomy_core.transformation.to_c import CppModel, CppNumerics
import zoomy_core.misc.misc as misc
from zoomy_core.model.custom_sympy_functions import conditional



## Model definition

In [ ]:
bcs = BC.BoundaryConditions(
    [
        BC.Wall(tag="default", momentum_field_indices=[[2, 4], [3, 5]]),
    ]
)

class SMEB(NumericalShallowMomentsTopo):
    def initial_condition(self):
        X = self.position
        p = self.parameters
        out = misc.ZArray.zeros(self.n_variables)
        out[0] = 1.
        out[1] = conditional(X[0] < 5, 2., 0.)
        return out

    def initial_aux_condition(self):
        X = self.position
        p = self.parameters
        eps = p.eps
        out = misc.ZArray.zeros(self.n_aux_variables)
        out[0] = conditional(X[0] < 5, 1./2., 1./eps)
        return out
    
    def update_variables(self):
        Q = self.variables
        h = Q[1]
        Qout = misc.ZArray(Q)
        h = conditional(h < 0, 0, h)
        Qout[1] = h
        eps = self.parameters.eps
        factor = h / (h + eps*100.)
        for i in range(2, self.n_variables):
            Qout[i] *= factor
        return Qout
    
    
    def update_aux_variables(self):
        Q = self.variables
        Qaux = misc.ZArray(self.aux_variables)
        eps = self.parameters.eps
        Qaux[0] = Q[1] / (Q[1]**2 + eps)
        return misc.ZArray(Qaux)
    
    def source(self):
        return self.slip()
    
    

model = SMEB(
    level=1 ,
    dimension=2,
    boundary_conditions=bcs,
    aux_variables=2,
)


## Sympy Model

In [10]:
import sympy as sp

## Code transformation

In [16]:
settings = Settings(name="ShallowWater", output=Zstruct(directory="outputs/trafo", filename="swe.h5", clean_directory=True, snapshots=30))
CppModel.write_code(model, settings)


'/home/ingo/Git/Zoomy/outputs/trafo/.c_interface/Model.H'

In [17]:
! cp /home/ingo/Git/Zoomy/outputs/trafo/.c_interface/Model.H /home/ingo/Git/Zoomy/library/zoomy_dmplex

In [18]:
class Numerics(PositiveNonconservativeRusanov):
    def get_viscosity_identity_flux(self):
        Id = sp.Matrix(sp.Identity(self.model.n_variables))
        lvl = self.model.level
        offset = lvl+1
        Id = 0 * Id
        Id[1, 1] = 1
        Id[2, 2] = 1
        Id[2+offset, 2+offset] = 1
        return misc.ZArray(Id)
    
    def get_viscosity_identity_fluctuations(self):
        Id = sp.Matrix(sp.Identity(self.model.n_variables))
        Id[0,0] = 0
        lvl = self.model.level
        offset = lvl+1
        Id = 0 * Id
        Id[1, 1] = 0
        Id[2, 2] = 0
        Id[2+offset, 2+offset] = 0
        return misc.ZArray(Id)
numerics = PositiveNonconservativeRusanov(model)

In [22]:
CppNumerics.write_code(numerics, settings)

'/home/ingo/Git/Zoomy/outputs/trafo/.c_interface/Numerics.H'

In [24]:
! cp /home/ingo/Git/Zoomy/outputs/trafo/.c_interface/Numerics.H /home/ingo/Git/Zoomy/library/zoomy_dmplex


## Check the output

In [25]:
main_dir = misc.get_main_directory()
path = os.path.join(main_dir, os.path.join(settings.output.directory, '.c_interface/Model.H'))
with open(path, "r") as f:
    print(f.read())


#pragma once
#include <cmath>
#include <array>

#ifndef ZOOMY_SIMPLE_ARRAY
#define ZOOMY_SIMPLE_ARRAY
template <typename T, int N>
struct SimpleArray {
    T data[N];
    T& operator[](int i) { return data[i]; }
    const T& operator[](int i) const { return data[i]; }
    T* begin() { return data; }
    const T* begin() const { return data; }
    T* end() { return data + N; }
    const T* end() const { return data + N; }
};
#endif

#include <vector>
#include <string>
#include <algorithm>

#ifdef __CUDACC__
#define PORTABLE_FN __host__ __device__
#else
#define PORTABLE_FN
#endif

template <typename T>
struct Model {
    static constexpr int n_dof_q    = 6;
    static constexpr int n_dof_qaux = 1;
    static constexpr int dimension  = 2;
    static constexpr int n_boundary_tags = 1;
    static const std::vector<std::string> get_boundary_tags() { return { "default" }; }
    static const std::vector<std::string> parameter_names() { return { "g", "chezy_C", "eps", "ex", "ey", "ez", "rho", "

In [26]:
main_dir = misc.get_main_directory()
path = os.path.join(
    main_dir, os.path.join(settings.output.directory, ".c_interface/Numerics.H")
)
with open(path, "r") as f:
    print(f.read())


#pragma once
#include <cmath>
#include <array>
#include "Model.H"

#ifndef ZOOMY_SIMPLE_ARRAY
#define ZOOMY_SIMPLE_ARRAY
template <typename T, int N>
struct SimpleArray {
    T data[N];
    T& operator[](int i) { return data[i]; }
    const T& operator[](int i) const { return data[i]; }
    T* begin() { return data; }
    const T* begin() const { return data; }
    T* end() { return data + N; }
    const T* end() const { return data + N; }
};
#endif

#include <vector>
#include <algorithm>

#ifdef __CUDACC__
#define PORTABLE_FN __host__ __device__
#else
#define PORTABLE_FN
#endif

template <typename T>
struct Numerics {
    static constexpr int n_dof_q = 6;
    PORTABLE_FN static inline SimpleArray<T, 6> numerical_flux(
        const T* Q_minus,
        const T* Q_plus,
        const T* Qaux_minus,
        const T* Qaux_plus,
        const T* p,
        const T* n)
    {
    T t0 = std::max(Q_minus[0], Q_plus[0]);
    T t1 = -t0;
    T t2 = std::max(0.0, Q_minus[0] + Q_minus[1] + t1);
 

In [27]:
main_dir = misc.get_main_directory()
path = os.path.join(
    main_dir, os.path.join(main_dir, "library/zoomy_dmplex/Numerics.H")
)
with open(path, "r") as f:
    print(f.read())


#pragma once
#include <cmath>
#include <array>
#include "Model.H"

#ifndef ZOOMY_SIMPLE_ARRAY
#define ZOOMY_SIMPLE_ARRAY
template <typename T, int N>
struct SimpleArray {
    T data[N];
    T& operator[](int i) { return data[i]; }
    const T& operator[](int i) const { return data[i]; }
    T* begin() { return data; }
    const T* begin() const { return data; }
    T* end() { return data + N; }
    const T* end() const { return data + N; }
};
#endif

#include <vector>
#include <algorithm>

#ifdef __CUDACC__
#define PORTABLE_FN __host__ __device__
#else
#define PORTABLE_FN
#endif

template <typename T>
struct Numerics {
    static constexpr int n_dof_q = 6;
    PORTABLE_FN static inline SimpleArray<T, 6> numerical_flux(
        const T* Q_minus,
        const T* Q_plus,
        const T* Qaux_minus,
        const T* Qaux_plus,
        const T* p,
        const T* n)
    {
    T t0 = std::max(Q_minus[0], Q_plus[0]);
    T t1 = -t0;
    T t2 = std::max(0.0, Q_minus[0] + Q_minus[1] + t1);
 